In [0]:

import json
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp

spark = SparkSession.builder.getOrCreate()

# Path to your managed volume
src_path = "/Volumes/0725catalog/src/data"

file_path = "/Volumes/0725catalog/src/data/processed_files.json"

# List files in volume folder
all_files = [f.path for f in dbutils.fs.ls("/Volumes/0725catalog/src/data")]
all_files = [f for f in all_files if not f.endswith(".json")]

# Check if tracking file exists and load processed files
if os.path.exists(file_path):
    with open(file_path, "r") as f:
        processed_files = json.load(f)
else:
    processed_files = []

print("Already processed files:")
print(processed_files)

# Filter to only new (unprocessed) files
new_files = [f for f in all_files if f not in processed_files]

if new_files:
    print("New files to process:")
    print(new_files)

    # Load and process new files
    #df = spark.read.format("csv").option("header", "true").load(new_files)
    
    # TODO: Add your processing logic here
    # e.g., df.write.format(...).save(...)
    # Read CSV from volume (simulating Auto Loader)
    df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .load(new_files)
        .withColumn("ingest_time", current_timestamp())  # optional tracking column
)

    # Append to Bronze Delta table
    df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("0725catalog.bronze.customers_raw")
    # Update tracking list with newly processed files
    processed_files.extend(new_files)

    # Save updated list
    with open(file_path, "w") as f:
        json.dump(processed_files, f)

    print("Updated processed files list:")
    print(processed_files)
else:
    print("No new files to process.")